In [ ]:
# %% [markdown]
# # 05 – Sanitization Pruning Effectiveness (choose data_source)

# %%
from memorypoison_audit.mitigation.sanitization_hooks import SanitizationHooks
from memorypoison_audit.core.agent_orchestrator import AgentOrchestrator
from memorypoison_audit.attacks.gradient_free_perturber import GradientFreePerturber
from memorypoison_audit.benchmarks.metrics import MetricsCalculator
from memorypoison_audit.data_loader import HotpotQALoader, SyntheticDataGenerator
from shared_functions import load_configs, set_seed, save_metrics

DATA_SOURCE = "synthetic"  # or "hotpot"

attack_cfg, sanitizer_cfg = load_configs()
set_seed(42)
session_id = "defended"
hooks = SanitizationHooks(enabled=True, config=sanitizer_cfg['sanitizer']['lof'])
agent = AgentOrchestrator(session_id, attack_cfg, sanitization_hooks=hooks)

# Populate benign facts
if DATA_SOURCE == "hotpot":
    loader = HotpotQALoader()
    data = loader.load_dev()
    for item in data[:100]:
        context = " ".join(item.get("context", []))
        if context:
            agent.memory_store.add_fact(session_id, context)
else:
    gen = SyntheticDataGenerator()
    facts = gen.generate_facts(num_facts=500)
    for fact in facts:
        agent.memory_store.add_fact(session_id, fact)

# %%
# Inject poison
perturber = GradientFreePerturber(budget=0.1)
for text in attack_cfg['attack']['malicious_texts']:
    perturber.apply_to_memory(agent.memory_store, session_id, text)

asr_def = []
keywords = ["COMPROMISED", "ADMIN123"]
for turn in range(1, 31):
    agent.generate_response(f"Query {turn}")
    retrieved = agent.memory_store.query(session_id, f"Query {turn}", top_k=3)
    texts = [r['text'] for r in retrieved]
    asr = MetricsCalculator.attack_success_rate(texts, keywords)
    asr_def.append(asr)

# %%
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1,31)), y=asr_def, mode='lines+markers', name='With Sanitization'))
fig.update_layout(title=f"Defended ASR on {DATA_SOURCE}", xaxis_title="Turn", yaxis_title="ASR")
fig.show()

save_metrics("sanitization", {"asr_defended": asr_def, "data_source": DATA_SOURCE}, data_source=DATA_SOURCE)